In [ ]:
import os, re, pandas as pd, numpy as np
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab')
    nltk.download('stopwords', quiet=True)
except:
    print("Warning: Could not download NLTK data")

print("="*60)
print("PHÂN TÍCH TẬP DỮ LIỆU CRANFIELD")
print("="*60)

print("\n[1] ĐANG TẢI DỮ LIỆU...")

# Load Documents từ thư mục Cranfield/Cranfield/
docs = []
if os.path.exists('Cranfield/Cranfield'):
    for f in sorted([f for f in os.listdir('Cranfield/Cranfield') if f.endswith('.txt')],
                   key=lambda x: int(x.split('.')[0])):
        with open(f'Cranfield/Cranfield/{f}', 'r', encoding='utf-8', errors='ignore') as file:
            docs.append({
                'id': int(f.split('.')[0]),
                'text': file.read().strip()
            })
else:
    print("Không tìm thấy thư mục 'Cranfield/Cranfield/'")

# Load Queries từ query.txt
queries = []
if os.path.exists('Cranfield/query.txt'):
    with open('Cranfield/query.txt', 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            if line.strip() and '\t' in line:
                qid, qtext = line.strip().split('\t', 1)
                queries.append({'id': int(qid), 'text': qtext})
else:
    print("Không tìm thấy file 'query.txt'")

# Load Relevance Judgments từ RES/
rel_data = []
if os.path.exists('Cranfield/RES'):
    res_files = [f for f in os.listdir('Cranfield/RES') if f.endswith('.txt')]
    if res_files:
        for res_file in res_files:
            query_id = int(res_file.split('.')[0])  # "1.txt" → 1

            with open(f'Cranfield/RES/{res_file}', 'r', encoding='utf-8', errors='ignore') as file:
                for line in file:
                    parts = line.strip().split('\t')
                    if len(parts) < 2:
                        continue

                    doc_id = int(parts[0].split()[1])  # "217 666" → 666
                    relevance = int(parts[1])

                    rel_data.append({
                        'query_id': query_id,
                        'doc_id': doc_id,
                        'relevance': relevance
                    })
else:
    print("Không tìm thấy thư mục 'RES/'")


print(f"✓ Documents: {len(docs)}")
print(f"✓ Queries: {len(queries)}")
print(f"✓ Relevance judgments: {len(rel_data)}")


PHÂN TÍCH TẬP DỮ LIỆU CRANFIELD

[1] ĐANG TẢI DỮ LIỆU...
✓ Documents: 1400
✓ Queries: 225
✓ Relevance judgments: 1836


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [47]:
docs[:2]

[{'id': 1,
  'text': 'experimental investigation of the aerodynamics of a wing in a slipstream . an experimental study of a wing in a propeller slipstream was made in order to determine the spanwise distribution of the lift increase due to slipstream at different angles of attack of the wing and at different free stream to slipstream velocity ratios .  the results were intended in part as an evaluation basis for different theoretical treatments of this problem . the comparative span loading curves, together with supporting evidence, showed that a substantial part of the lift increment produced by the slipstream was due to a /destalling/ or boundary layer control effect .  the integrated remaining lift increment, after subtracting this destalling lift, was found to agree well with a potential flow theory . an empirical evaluation of the destalling effects was made for the specific configuration of the experiment .'},
 {'id': 2,
  'text': "simple shear flow past a flat plate in an incomp

In [48]:
queries[:10]

[{'id': 1,
  'text': 'what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .'},
 {'id': 2,
  'text': 'what are the structural and aeroelastic problems associated with flight of high speed aircraft .'},
 {'id': 3,
  'text': 'what problems of heat conduction in composite slabs have been solved so far .'},
 {'id': 4,
  'text': 'can a criterion be developed to show empirically the validity of flow solutions for chemically reacting gas mixtures based on the simplifying assumption of instantaneous local chemical equilibrium .'},
 {'id': 5,
  'text': 'what chemical kinetic system is applicable to hypersonic aerodynamic problems .'},
 {'id': 6,
  'text': 'what theoretical and experimental guides do we have as to turbulent couette flow behaviour .'},
 {'id': 7,
  'text': 'is it possible to relate the available pressure distributions for an ogive forebody at zero angle of attack to the lower surface pressures of an equivalent ogive forebody at an

In [49]:
rel_data[:10]

[{'query_id': 16, 'doc_id': 266, 'relevance': 2},
 {'query_id': 16, 'doc_id': 106, 'relevance': 3},
 {'query_id': 16, 'doc_id': 196, 'relevance': 3},
 {'query_id': 16, 'doc_id': 498, 'relevance': -1},
 {'query_id': 215, 'doc_id': 37, 'relevance': 4},
 {'query_id': 215, 'doc_id': 35, 'relevance': 4},
 {'query_id': 215, 'doc_id': 535, 'relevance': -1},
 {'query_id': 83, 'doc_id': 680, 'relevance': 2},
 {'query_id': 83, 'doc_id': 681, 'relevance': 2},
 {'query_id': 83, 'doc_id': 682, 'relevance': 2}]

In [51]:
print("\n[2] THỐNG KÊ CƠ BẢN")
print("-" * 60)

# Documents
if docs:
    doc_df = pd.DataFrame(docs)
    doc_df['word_count'] = doc_df['text'].apply(lambda x: len(x.split()))
    doc_df['char_count'] = doc_df['text'].apply(len)

    print(f"DOCUMENTS:")
    print(f"• Tổng số: {len(doc_df)}")
    print(f"• Trung bình: {doc_df['word_count'].mean():.0f} từ/doc")
    print(f"• Khoảng: {doc_df['word_count'].min()}-{doc_df['word_count'].max()} từ")
    print(f"• Median: {doc_df['word_count'].median():.0f} từ")

# Queries
if queries:
    query_df = pd.DataFrame(queries)
    query_df['word_count'] = query_df['text'].apply(lambda x: len(x.split()))

    print(f"\nQUERIES:")
    print(f"• Tổng số: {len(query_df)}")
    print(f"• Trung bình: {query_df['word_count'].mean():.1f} từ/query")
    print(f"• Khoảng: {query_df['word_count'].min()}-{query_df['word_count'].max()} từ")

# Relevance Judgments
if rel_data:
    rel_df = pd.DataFrame(rel_data)

    print(f"\nRELEVANCE JUDGMENTS:")
    print(f"• Tổng số: {len(rel_df)} đánh giá")
    print(f"• Queries có đánh giá: {rel_df['query_id'].nunique()}")
    print(f"• Trung bình docs/query: {rel_df.groupby('query_id').size().mean():.1f}")
    print(f"• Điểm relevance: {sorted(rel_df['relevance'].unique())}")

    # Phân bố relevance
    rel_dist = rel_df['relevance'].value_counts().sort_index()
    print(f"\nPhân bố điểm relevance:")
    for score, count in rel_dist.items():
        print(f"• Score {score}: {count} ({count/len(rel_df)*100:.1f}%)")



[2] THỐNG KÊ CƠ BẢN
------------------------------------------------------------
DOCUMENTS:
• Tổng số: 1400
• Trung bình: 168 từ/doc
• Khoảng: 0-679 từ
• Median: 148 từ

QUERIES:
• Tổng số: 225
• Trung bình: 18.0 từ/query
• Khoảng: 6-46 từ

RELEVANCE JUDGMENTS:
• Tổng số: 1836 đánh giá
• Queries có đánh giá: 225
• Trung bình docs/query: 8.2
• Điểm relevance: [np.int64(-1), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Phân bố điểm relevance:
• Score -1: 225 (12.3%)
• Score 1: 128 (7.0%)
• Score 2: 387 (21.1%)
• Score 3: 733 (39.9%)
• Score 4: 363 (19.8%)


In [53]:
print("\n[3] KIỂM TRA CHẤT LƯỢNG")
print("-" * 60)

issues_found = False

if docs:
    # Empty documents
    empty_docs = [d for d in docs if not d['text'].strip()]
    if empty_docs:
        empty_ids = [d['id'] for d in empty_docs]
        print(f"{len(empty_docs)} documents rỗng:")
        print(f"IDs: {empty_ids[:10]}")
        if len(empty_ids) > 10:
            print(f"... và {len(empty_ids) - 10} documents khác")
        issues_found = True

    # Very short documents
    very_short = doc_df[doc_df['word_count'] < 10]
    if len(very_short) > 0:
        short_ids = very_short['id'].tolist()
        print(f"{len(very_short)} documents có ít hơn 10 từ:")
        print(f"IDs: {short_ids[:10]}")
        if len(short_ids) > 10:
            print(f"... và {len(short_ids) - 10} documents khác")
        issues_found = True

if queries and rel_data:
    # Queries without relevance judgments
    query_ids_in_rel = set(rel_df['query_id'])
    query_ids_all = set([q['id'] for q in queries])
    missing = query_ids_all - query_ids_in_rel
    if missing:
        print(f"{len(missing)} queries không có relevance judgments:")
        print(f"Query IDs: {sorted(list(missing))[:10]}")
        if len(missing) > 10:
            print(f"... và {len(missing) - 10} queries khác")
        issues_found = True

    # Documents in relevance but not in collection
    doc_ids_in_rel = set(rel_df['doc_id'])
    doc_ids_all = set([d['id'] for d in docs])
    missing_docs = doc_ids_in_rel - doc_ids_all
    if missing_docs:
        print(f"{len(missing_docs)} documents được đánh giá nhưng không có trong collection:")
        print(f"Doc IDs: {sorted(list(missing_docs))[:10]}")
        if len(missing_docs) > 10:
            print(f"... và {len(missing_docs) - 10} documents khác")
        issues_found = True

if not issues_found:
    print("Không phát hiện vấn đề về chất lượng dữ liệu")



[3] KIỂM TRA CHẤT LƯỢNG
------------------------------------------------------------
2 documents rỗng:
IDs: [471, 995]
2 documents có ít hơn 10 từ:
IDs: [471, 995]


In [54]:
print("\n[4] TIỀN XỬ LÝ VĂN BẢN")
print("-" * 60)

def preprocess_text(text):
    """Tiền xử lý văn bản: lowercase, tokenize, remove stopwords, stemming"""
    # Lowercase
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove non-alphabetic
    tokens = [w for w in tokens if w.isalpha()]
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words]
    # Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(tokens)

if docs:
    print("Đang xử lý documents...")
    doc_df['processed'] = doc_df['text'].apply(preprocess_text)
    avg_tokens = doc_df['processed'].apply(lambda x: len(x.split())).mean()
    print(f"✓ Documents đã được xử lý")
    print(f"✓ Trung bình sau xử lý: {avg_tokens:.0f} tokens/doc")

if queries:
    print("Đang xử lý queries...")
    query_df['processed'] = query_df['text'].apply(preprocess_text)
    avg_tokens = query_df['processed'].apply(lambda x: len(x.split())).mean()
    print(f"✓ Queries đã được xử lý")
    print(f"✓ Trung bình sau xử lý: {avg_tokens:.0f} tokens/query")




[4] TIỀN XỬ LÝ VĂN BẢN
------------------------------------------------------------
Đang xử lý documents...
✓ Documents đã được xử lý
✓ Trung bình sau xử lý: 91 tokens/doc
Đang xử lý queries...
✓ Queries đã được xử lý
✓ Trung bình sau xử lý: 10 tokens/query


In [56]:
print("\n[5] PHÂN TÍCH TỪ VỰNG")
print("-" * 60)

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(doc_df['processed'])
word_counts = X.sum(axis=0).A1
features = vectorizer.get_feature_names_out()

sorted_idx = word_counts.argsort()[::-1]

if docs:
    print("TOP TỪ XUẤT HIỆN NHIỀU NHẤT TRONG DOCUMENTS (sau tiền xử lý):")
    for idx in sorted_idx[:10]:
        print(f"{features[idx]}: {word_counts[idx]}")


[5] PHÂN TÍCH TỪ VỰNG
------------------------------------------------------------
TOP TỪ XUẤT HIỆN NHIỀU NHẤT TRONG DOCUMENTS (sau tiền xử lý):
flow: 2078
pressur: 1389
number: 1348
boundari: 1215
layer: 1160
result: 1087
effect: 994
method: 883
theori: 881
bodi: 852


In [58]:
print("\n[6] PHÂN TÍCH TF-IDF")
print("-" * 60)

if docs and len(doc_df) > 0:
    try:
        tfidf = TfidfVectorizer(max_features=None)
        tfidf_matrix = tfidf.fit_transform(doc_df['processed'])

        feature_names = tfidf.get_feature_names_out()
        dense = tfidf_matrix.todense()
        term_scores = dense.sum(axis=0).A1
        sorted_idx = term_scores.argsort()[::-1]

        #print(f"Số lượng từ vựng: {len(feature_names)}")
        #print(f"Kích thước ma trận: {tfidf_matrix.shape}")

        print("TOP TERMS THEO TF-IDF SCORE:")
        for idx in sorted_idx[:15]:
            print(f"{feature_names[idx]:15} : {term_scores[idx]:8.4f}")
    except Exception as e:
        print(f"Không thể tính TF-IDF: {str(e)}")




[6] PHÂN TÍCH TF-IDF
------------------------------------------------------------
TOP TERMS THEO TF-IDF SCORE:
flow            :  69.4630
boundari        :  50.6975
layer           :  50.3420
pressur         :  49.9637
number          :  46.8280
wing            :  42.3122
heat            :  41.8490
bodi            :  40.3139
solut           :  39.6089
shock           :  38.3652
method          :  38.0925
theori          :  37.3947
equat           :  36.7108
result          :  36.3960
effect          :  35.5097


In [22]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 55.4 MB/s eta 0:00:00


In [60]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from gensim.corpora.dictionary import Dictionary
from gensim.models import LdaModel

In [61]:
print("\n[7] PHÂN TÍCH TOPIC (LDA)")
print("-" * 60)

if docs:
    # documents_preprocessed: list các document sau tiền xử lý
    processed_texts = doc_df['processed'].tolist()

    # Chuyển processed text thành list of tokens
    tokenized_docs = [text.split() for text in processed_texts]

    dictionary = Dictionary(tokenized_docs)
    corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_docs]

    print("Số lượng từ vựng:", len(dictionary))
    print("Số lượng document:", len(corpus_bow))

    num_topics = 5
    lda_model = LdaModel(
        corpus=corpus_bow,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=42,
        passes=10,
        alpha='auto'
    )

    for idx, topic in lda_model.print_topics(num_words=10):
        print(f"Topic {idx}: {topic}")



[7] PHÂN TÍCH TOPIC (LDA)
------------------------------------------------------------
Số lượng từ vựng: 4252
Số lượng document: 1400
Topic 0: 0.030*"layer" + 0.029*"boundari" + 0.019*"flow" + 0.015*"heat" + 0.012*"transfer" + 0.011*"laminar" + 0.011*"temperatur" + 0.010*"solut" + 0.009*"number" + 0.009*"wall"
Topic 1: 0.012*"problem" + 0.012*"buckl" + 0.011*"load" + 0.011*"shell" + 0.010*"solut" + 0.010*"theori" + 0.010*"stress" + 0.010*"equat" + 0.009*"result" + 0.009*"shock"
Topic 2: 0.019*"number" + 0.014*"mach" + 0.011*"effect" + 0.011*"test" + 0.010*"heat" + 0.009*"angl" + 0.009*"pressur" + 0.009*"result" + 0.009*"model" + 0.008*"flutter"
Topic 3: 0.025*"jet" + 0.025*"pressur" + 0.022*"flow" + 0.015*"number" + 0.014*"nozzl" + 0.011*"shock" + 0.011*"mach" + 0.009*"ratio" + 0.009*"effect" + 0.009*"ga"
Topic 4: 0.026*"flow" + 0.020*"wing" + 0.019*"bodi" + 0.012*"method" + 0.011*"pressur" + 0.011*"theori" + 0.010*"distribut" + 0.009*"result" + 0.009*"lift" + 0.009*"calcul"


In [62]:
print("\n[8] PHÂN TÍCH TƯƠNG TỰ (COSINE SIMILARITY)")
print("-" * 60)

if docs and len(doc_df) > 0:
    if 'tfidf_matrix' not in locals():
        print("Đang tính TF-IDF matrix...")
        from sklearn.feature_extraction.text import TfidfVectorizer
        tfidf = TfidfVectorizer(max_features=50)
        tfidf_matrix = tfidf.fit_transform(doc_df['processed'])

    # Tính cosine similarity
    similarity_matrix = cosine_similarity(tfidf_matrix)
    print("Kích thước ma trận similarity:", similarity_matrix.shape)

    # Lấy các giá trị trên tam giác trên (không tính đường chéo)
    sim_values = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

    print("Cosine similarity trung bình:", np.mean(sim_values))
    print("Cosine similarity lớn nhất:", np.max(sim_values))
    print("Cosine similarity nhỏ nhất:", np.min(sim_values))

    # Tìm cặp document có similarity cao nhất
    max_idx = np.argmax(sim_values)
    triu_indices = np.triu_indices_from(similarity_matrix, k=1)
    doc_i = triu_indices[0][max_idx]
    doc_j = triu_indices[1][max_idx]

    print("\nCặp document tương đồng nhất:")
    print("Document", doc_i, "và Document", doc_j)
    print("Cosine similarity:", similarity_matrix[doc_i, doc_j])
else:
    print("Không có documents để tính similarity")



[8] PHÂN TÍCH TƯƠNG TỰ (COSINE SIMILARITY)
------------------------------------------------------------
Kích thước ma trận similarity: (1400, 1400)
Cosine similarity trung bình: 0.055342607792940875
Cosine similarity lớn nhất: 0.975731693948919
Cosine similarity nhỏ nhất: 0.0

Cặp document tương đồng nhất:
Document 1273 và Document 1318
Cosine similarity: 0.975731693948919
